# 08 — Generalisation: can the DQN draw letters it has never seen?

The DQN of notebook 07 learned the 10 training letters with a single network that receives the target letter
as input. If it has learned to *write* in general — follow the segments of the target, never draw outside,
press STOP when everything is drawn — it should be able to draw a new letter. Here we test the **3 trained
networks, without any new training**, on the **16 other letters** of the alphabet.

**Hypothesis before the experiment:** letters that resemble training letters (P ≈ F, M and W ≈ N, Y ≈ V + I,
K ≈ X and H) should be drawn better than curved letters (C, O, S, G).

The networks must have been trained and saved by notebook 07 (`results/07_dqn_seed*.pt`).

In [ ]:
import sys
sys.path.insert(0, "..")

import csv
import numpy as np
import matplotlib.pyplot as plt
import torch
from IPython.display import Image

from src.letters import make_letter, TRAINING_LETTERS, TEST_LETTERS
from src.environment import LetterWritingEnv
from src.dqn import QNetwork, greedy_episode
from src.visualization import plot_letters, plot_episodes, animate_episodes, SURFACE, TEXT, MUTED

torch.set_num_threads(1)
GAMMA = 0.99
SEEDS = [0, 1, 2]
nets = []
for seed in SEEDS:
    net = QNetwork()
    net.load_state_dict(torch.load(f"../results/07_dqn_seed{seed}.pt"))
    net.eval()
    nets.append(net)
print(f"{len(nets)} trained networks loaded")


def style(ax, title=None, xlabel=None, ylabel=None):
    ax.set_facecolor(SURFACE)
    ax.grid(axis="y", color="#ecebe7", lw=0.8)
    ax.tick_params(colors=MUTED, labelsize=8)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_color("#c3c2b7")
    if title:
        ax.set_title(title, color=TEXT, fontsize=10)
    if xlabel:
        ax.set_xlabel(xlabel, color=MUTED, fontsize=9)
    if ylabel:
        ax.set_ylabel(ylabel, color=MUTED, fontsize=9)

## 1. The 16 test letters

Defined in `src/letters.py` like the training letters: strokes on the 5×5 grid (curves approximated by
straight and diagonal strokes) and natural starting points (orange).

In [ ]:
test_letters = [make_letter(name) for name in TEST_LETTERS]
plot_letters(test_letters, ncols=8, path="../results/08_test_letters.png")
plt.show()

## 2. How the networks draw the unseen letters

For each letter: greedy policy of each of the 3 networks, from each natural starting point. We measure:
- **perfect**: complete letter, no segment outside, then STOP;
- **coverage**: fraction of the segments of the letter drawn;
- **off**: segments drawn outside the letter (mean per episode);
- **STOP**: fraction of the episodes where the agent pressed STOP (instead of reaching the 50-step limit);
- **similarity**: overlap with the most similar training letter (Jaccard index = common segments / all segments of both letters).

Two references: the same measures on the **training letters** (what the networks can do), and a **random
agent** on the test letters (what "knowing nothing" gives).

In [ ]:
train_sets = {n: set(make_letter(n).segments) for n in TRAINING_LETTERS}

def measure(name):
    env = LetterWritingEnv(name)
    eps = [greedy_episode(net, env, start, GAMMA) for net in nets for start in env.letter.start_points]
    return {
        "perfect": np.mean([e["completed"] and e["off_letter"] == 0 for e in eps]),
        "coverage": np.mean([e["drawn"] / env.k for e in eps]),
        "off": np.mean([e["off_letter"] for e in eps]),
        "stop": np.mean([e["history"][-1]["event"] == "stop" for e in eps]),
        "k": env.k,
    }

def random_coverage(name, n=200, seed=0):
    env = LetterWritingEnv(name)
    rng = np.random.default_rng(seed)
    cov = []
    for _ in range(n):
        env.reset()
        done = False
        while not done:
            _, _, terminated, truncated, _ = env.step(int(rng.integers(env.n_actions)))
            done = terminated or truncated
        cov.append(env.drawn_segments() / env.k)
    return np.mean(cov)

def similarity(name):
    segs = set(make_letter(name).segments)
    return max((len(segs & t) / len(segs | t), n) for n, t in train_sets.items())

train_results = {name: measure(name) for name in TRAINING_LETTERS}
print("Training letters (reference)")
print(f"  perfect {np.mean([r['perfect'] for r in train_results.values()]):.0%}, "
      f"coverage {np.mean([r['coverage'] for r in train_results.values()]):.0%}, "
      f"STOP {np.mean([r['stop'] for r in train_results.values()]):.0%}")

rows = []
print(f"\n{'letter':<7}{'seg.':>5}{'perfect':>9}{'coverage':>10}{'random':>8}{'off':>6}{'STOP':>6}   most similar training letter")
for name in TEST_LETTERS:
    r = measure(name)
    rc = random_coverage(name)
    sim, closest = similarity(name)
    rows.append({"letter": name, "segments": r["k"], "perfect": r["perfect"], "coverage": round(r["coverage"], 3),
                 "random_coverage": round(rc, 3), "off_letter": round(r["off"], 2), "stop": r["stop"],
                 "closest_training_letter": closest, "jaccard": round(sim, 2)})
    print(f"{name:<7}{r['k']:>5}{r['perfect']:>9.0%}{r['coverage']:>10.0%}{rc:>8.0%}{r['off']:>6.1f}{r['stop']:>6.0%}   {closest} ({sim:.2f})")

with open("../results/08_generalisation.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=rows[0].keys())
    writer.writeheader()
    writer.writerows(rows)
print("\nsaved: results/08_generalisation.csv")

### Does similarity to a training letter help?

Coverage on each test letter as a function of its similarity with the closest training letter.

In [ ]:
sims = np.array([r["jaccard"] for r in rows])
covs = np.array([100 * r["coverage"] for r in rows])
fig, ax = plt.subplots(figsize=(6.5, 4), facecolor=SURFACE)
ax.scatter(sims, covs, s=60, color="#2a78d6", edgecolor=SURFACE, linewidth=1.5, zorder=3)
for i, (r, x, y) in enumerate(sorted(zip(rows, sims, covs), key=lambda t: (t[2], t[1]))):
    ax.annotate(f"{r['letter']} (~{r['closest_training_letter']})", (x, y), xytext=(6, 4 if i % 2 == 0 else -11),
                textcoords="offset points", fontsize=8, color=TEXT)
style(ax, "Unseen letters: coverage vs similarity to the closest training letter",
      "similarity (Jaccard index)", "segments drawn (%)")
ax.set_ylim(-5, 100)
fig.tight_layout()
fig.savefig("../results/08_similarity.png", dpi=150, facecolor=SURFACE)
plt.show()
print(f"correlation between similarity and coverage: {np.corrcoef(sims, covs)[0, 1]:.2f}")

## 3. What the network actually does on a new letter

Greedy episodes of network 0 on the 16 test letters, from the first starting point.

In [ ]:
panels = []
for name in TEST_LETTERS:
    env = LetterWritingEnv(name)
    ep = greedy_episode(nets[0], env, env.letter.start_points[0], GAMMA)
    end = "STOP" if ep["history"][-1]["event"] == "stop" else "50-step limit"
    panels.append((env.letter, ep["history"], f"{name}: {ep['drawn']}/{env.k} drawn, {end}"))
fig = plot_episodes(panels, ncols=8, path="../results/08_test_drawings.png")
for ax in fig.axes:
    ax.title.set_fontsize(8)
plt.show()
animate_episodes(panels[:8], "../results/08_test_drawings.gif", ncols=8)
Image(filename="../results/08_test_drawings.gif")

## What to notice

- **The DQN does not generalise.** On the 16 unseen letters: 0 % perfect drawings, only 0–33 % of the segments
  drawn, and the agent **never presses STOP** (all episodes reach the 50-step limit). On the training letters,
  the same networks reach 96 % perfect drawings, 99 % coverage and press STOP in 96 % of the episodes.
- **Barely better than random.** A random agent draws 5–8 % of the segments. The DQN does better on some
  letters (D, K, P, U: about 30 %) but worse on others (G, Q, Y: 0 %).
- **It replays memorised routines.** The best-covered letters start like a training letter: for D and K, the
  network draws the vertical stem from the top-left corner, as it does for L, E or F, and then gets lost,
  wanders with the pen up or draws outside the letter (A, O, P, U: 8–13 segments outside the letter on average).
  Similarity helps only weakly (correlation 0.30 between similarity and coverage): W is very similar to N
  (Jaccard 0.71) but only 6 % of it is drawn.
- **Why.** With only 10 different targets during training, the 72 "target" bits act as an *identifier* of the
  letter: the network learns 10 separate behaviours indexed by this identifier, which is enough to be perfect on
  the training set, instead of the general rule "draw the target segments that are still missing, then STOP when
  the drawn bits equal the target bits". Nothing forces it to learn the general rule from 10 examples. A new
  letter is an input it has never seen, and its outputs are meaningless — including the STOP decision. This is
  a known problem of reinforcement learning: agents overfit to their training tasks, and need many diverse
  training tasks to generalise (Cobbe et al., 2019, *Quantifying generalization in reinforcement learning*).

**For the debate:** the DQN generalises *between the states of the letters it was trained on* (notebook 07: it is
closer to the optimum than the tables, with far fewer episodes), but not *to new letters*. "Being able to take
the letter as input" is not the same as "having learned to write".

**Possible improvements** (not done here):
1. **Many more, diverse targets**: train on thousands of random stroke figures generated automatically, then
   test on the real letters, so that memorising each target becomes impossible.
2. **A more helpful input or architecture**: give the network the *missing* segments (target minus drawn)
   directly, or use a convolutional network on an image of the grid, so that a local rule ("there is a missing
   segment next to the pen") is shared between positions and letters.
3. **More training letters**: e.g. train on 20 letters and test on the remaining 6.
